In [22]:
import tensorflow as tf 
import numpy as np 
class IntentClassificationModel(tf.keras.Model):
    def __init__(self, num_classes):
        super().__init__()
        self.dense1=tf.keras.layers.Dense(128,activation="relu")
        self.dropout1=tf.keras.layers.Dropout(0.3)

        self.dense2 =tf.keras.layers.Dense(64,activation='relu')
        self.dropout2=tf.keras.layers.Dropout(0.2)
        self.output_layer=tf.keras.layers.Dense(num_classes,activation='softmax')
    def call(self, inputs, training =False):
        x =self.dense1(inputs)
        x= self.dropout1(x,training=training)

        x =self.dense2(x)
        x =self.dropout2(x,training=training)

        return self.output_layer(x)

In [23]:
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder


class SentenceTransformerIntentClassifier:
    def __init__(self, sentence_model_name='sentence-transformers/all-MiniLM-L6-v2'):
        self.sentence_model_name =sentence_model_name
        self.sentence_model =SentenceTransformer(self.sentence_model_name)
        self.label_encoder=LabelEncoder()


        self.model =None 
        self.input_dim =None 
        self.num_classes =None 


    def normalize(self,v):
        return v/np.linalg.norm(v)
    
    
    def encode_texts(self, texts):
        # embeedings =self.sentence_model.encode(
        #     texts ,
        #     convert_to_numpy=True , 
        #     normalize_embedding=True 
        # )
        embeedings =self.sentence_model.encode(texts)
        embeedings=self.normalize(embeedings)
        return embeedings.astype(np.float32)
    
    def fit (self, texts , labels , epochs =50, batch_size =8):
        texts =list(texts )
        labels =list(labels)

        y =self.label_encoder.fit_transform(labels).astype(np.int32)

        x =self.encode_texts(texts)

        self.input_dim =x.shape[1]
        self.num_classes=len(self.label_encoder.classes_)

        print("Embedding shape :", x.shape)
        print("Classes :", self.label_encoder.classes_)
        print("Number of classes :", self.num_classes)
        print("Unique endcoded labels :",np.unique(y))


        tf.keras.backend.clear_session()

        self.model=IntentClassificationModel(num_classes=self.num_classes)
        self.model.build(input_shape=(None , self.input_dim))
        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate =0.001),
            loss ="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )

        self.model.summary()

        history=self.model.fit(x,y)

        return history
    

    def predict(self, text ):
        if isinstance(text , str ):
            texts =[text]
        else:
            texts= list(text)
        
        x =self.encode_texts(texts)

        probs =self.model.predict(x, verbose=0)

        predict_ids = np.argmax(probs , axis =1)

        predicted_labels =self.label_encoder.inverse_transform(predict_ids)
        confidence_score = np.max(probs, axis =1)

        results=[]
#  "probablities": {
#                     class_name :float(score) for class_name, score in zip(self.label_encoder.classes_, probs)
#                 }
        for original_text, label , confidence , prob in zip(texts , predicted_labels, confidence_score, probs):
            results.append({
                "text": original_text, 
                "intent":label, 
                "confidence":float(confidence),
               
            })
        return results
    

In [24]:
import yaml
import tensorflow as tf
from pathlib import Path 
intent_file=Path("../../data/intent-classifier.yml")
INTENTS={}
with open(intent_file, 'r', encoding='utf-8') as file:
    INTENTS=yaml.safe_load(file)


print(INTENTS)

{'cancel_order': ['cancel my order', 'I want to cancel my order', 'please cancel this order', 'stop my order', 'stop processing my order', 'delete my order', 'remove my order', 'terminate my order immediately', "I don't want this order anymore", 'I changed my mind about this order', "I don't want to continue with this order", "I don't want to move forward with this order", "please don't process this order", 'forget about my order', 'cancel it', 'stop it', 'never mind, cancel the order', 'I no longer need this order', 'can you cancel my purchase', 'withdraw my order request', "I don't want to move with this order", "I don't want to move forward with this order", "I don't want to continues with this order"], 'information_request': ['how does this work', 'what is your service', 'explain the process', 'tell me more details', 'give me more information', 'more information about delivery', 'information about payment', 'how can I use this service', 'what are the available options', 'tell me ab

In [25]:
labels =[]
texts =[]
for intent in INTENTS:
    for text in INTENTS[intent]:
        texts.append(text)
        labels.append(intent)



In [37]:
# classifier =SentenceTransformerIntentClassifier()
# history= classifier.fit(
#     texts =texts,
#     labels=labels
# )



In [39]:
label_encoder =LabelEncoder()
y =label_encoder.fit_transform(labels).astype(np.int32)
print("Classes :",label_encoder.classes_)
print("Size of intents :",len(y))
print("Maximum intent index :", np.max(y))
print("Unique indexs :", np.unique(y))

Classes : ['cancel_order' 'creation_request' 'information_request' 'order_creation']
Size of intents : 94
Maximum intent index : 3
Unique indexs : [0 1 2 3]


In [ ]:
sentence_transformer =SentenceTransformer("all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Help on SentenceTransformer in module sentence_transformers.sentence_transformer.model object:

class SentenceTransformer(sentence_transformers.base.model.BaseModel, sentence_transformers.sentence_transformer.fit_mixin.FitMixin)
 |  SentenceTransformer(
 |      model_name_or_path: 'str | None' = None,
 |      *,
 |      modules: 'list[nn.Module] | None' = None,
 |      device: 'str | None' = None,
 |      prompts: 'dict[str, str] | None' = None,
 |      default_prompt_name: 'str | None' = None,
 |      cache_folder: 'str | None' = None,
 |      trust_remote_code: 'bool' = False,
 |      revision: 'str | None' = None,
 |      local_files_only: 'bool' = False,
 |      token: 'bool | str | None' = None,
 |      use_auth_token: 'bool | str | None' = None,
 |      model_kwargs: 'dict[str, Any] | None' = None,
 |      processor_kwargs: 'dict[str, Any] | None' = None,
 |      config_kwargs: 'dict[str, Any] | None' = None,
 |      model_card_data: 'SentenceTransformerModelCardData | None' = No

In [41]:
x =sentence_transformer.encode(texts, convert_to_numpy=True, normalize_embeddings=False,show_progress_bar=True).astype(np.float32)
print("Embedding shape :", x.shape)
print("Label shape : ", y.shape)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embedding shape : (94, 384)
Label shape :  (94,)


In [44]:
from sklearn.preprocessing import StandardScaler


scaler =StandardScaler()

x_scaled =scaler.fit_transform(x).astype(np.float32)

num_classes =len(label_encoder.classes_)
input_dim = x.shape[1]

model =tf.keras.Sequential([
    tf.keras.layers.Input(shape=(input_dim,)),
    tf.keras.layers.Dense(128,activation='relu'),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(64,activation='relu'),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(num_classes,activation='softmax')
])

help(model.compile)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss ="sparse_categorical_crossentropy",
    metrics=['accuracy']
)
model.summary()

Help on method compile in module keras.src.trainers.trainer:

compile(
    optimizer='rmsprop',
    loss=None,
    loss_weights=None,
    metrics=None,
    weighted_metrics=None,
    run_eagerly=False,
    steps_per_execution=1,
    jit_compile='auto',
    auto_scale_loss=True
) method of keras.src.models.sequential.Sequential instance
    Configures the model for training.

    Example:

    ```python
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[
            keras.metrics.BinaryAccuracy(),
            keras.metrics.FalseNegatives(),
        ],
    )
    ```

    Args:
        optimizer: String (name of optimizer) or optimizer instance. See
            `keras.optimizers`.
        loss: Loss function. May be a string (name of loss function), or
            a `keras.losses.Loss` instance. See `keras.losses`. A
            loss function is any callable with the signature
            `loss =

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 128)            │        49,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 57,796 (225.77 KB)

 Trainable params: 57,796 (225.77 KB)

 Non-trainable params: 0 (0.00 B)

In [48]:
text ="I want to purchase this and complete the order"

embedding_text =sentence_transformer.encode([text],convert_to_numpy=True , normalize_embeddings=False).astype(np.float32)
embedding_text=scaler.transform(embedding_text).astype(np.float32)
probs =model.predict(embedding_text,verbose=0)
print(probs)
print(np.argmax(probs))

[[0.45553273 0.23481512 0.07273553 0.23691663]]
0
